**PERHITUNGAN PDRB HIJAU (2010-2025)**

PDRB Hijau = PDRB ADHK - Nilai Deforestasi - Nilai PM2.5

Sumber:
  1. PDRB_ADHK_Rill.csv                          -- PDRB ADHK, Rupiah penuh
  2. panel_deforestasi_nilai_rupiah_2010_2025.csv -- nilai_deforestasi_rp_3pct (utama)
  3. panel_nilai_pm25_lengkap_2010_2025.csv       -- nilai_pm25_rp_daly_1x (utama)

Tiga skenario PDRB Hijau dihasilkan sekaligus untuk uji sensitivitas:
  - Utama       : deforestasi (SCC 3%) + PM2.5 (DALY 1x)
  - Atas (tinggi kerugian): deforestasi (SCC 2,5%) + PM2.5 (DALY 3x)
  - VSL         : deforestasi (SCC 3%) + PM2.5 (VSL)


In [1]:
import pandas as pd
import re
from google.colab import drive

drive.mount('/content/drive')
FOLDER_SUMBER = "/content/drive/MyDrive/DatabaseSkripsi/SumberData"
FOLDER_OUTPUT = "/content/drive/MyDrive/DatabaseSkripsi/Outputs"

Mounted at /content/drive


In [ ]:
# =====================================================================
# 1. HARMONISASI
# =====================================================================
PROVINSI_STANDAR = [
    "ACEH", "BALI", "BANTEN", "BENGKULU", "DI YOGYAKARTA", "DKI JAKARTA",
    "GORONTALO", "JAMBI", "JAWA BARAT", "JAWA TENGAH", "JAWA TIMUR",
    "KALIMANTAN BARAT", "KALIMANTAN SELATAN", "KALIMANTAN TENGAH",
    "KALIMANTAN TIMUR", "KALIMANTAN UTARA", "KEP. BANGKA BELITUNG",
    "KEP. RIAU", "LAMPUNG", "MALUKU", "MALUKU UTARA",
    "NUSA TENGGARA BARAT", "NUSA TENGGARA TIMUR", "PAPUA", "PAPUA BARAT",
    "RIAU", "SULAWESI BARAT", "SULAWESI SELATAN", "SULAWESI TENGAH",
    "SULAWESI TENGGARA", "SULAWESI UTARA", "SUMATERA BARAT",
    "SUMATERA SELATAN", "SUMATERA UTARA",
]
PEMETAAN_KHUSUS = {
    "DAERAH ISTIMEWA YOGYAKARTA": "DI YOGYAKARTA", "YOGYAKARTA": "DI YOGYAKARTA",
    "KEPULAUAN BANGKA BELITUNG": "KEP. BANGKA BELITUNG",
    "KEPULAUAN BANGKA-BELITUNG": "KEP. BANGKA BELITUNG",
    "BANGKA BELITUNG": "KEP. BANGKA BELITUNG",
    "KEPULAUAN RIAU": "KEP. RIAU",
    "RIAU KEPULAUAN": "KEP. RIAU",
    "JAKARTA RAYA": "DKI JAKARTA", "JAKARTA": "DKI JAKARTA",
    "SUMATRA BARAT": "SUMATERA BARAT", "SUMATRA SELATAN": "SUMATERA SELATAN",
    "SUMATRA UTARA": "SUMATERA UTARA",
}


def harmonisasi_provinsi(nama):
    nama_upper = re.sub(r"\s+", " ", str(nama).strip()).upper()
    if nama_upper in PEMETAAN_KHUSUS:
        return PEMETAAN_KHUSUS[nama_upper]
    for standar in PROVINSI_STANDAR:
        if nama_upper == standar.upper():
            return standar
    return None

In [ ]:
# =====================================================================
# 2. READ SUMBER PEMBENTUKAN PDRB HIJAU
# =====================================================================
pdrb = pd.read_csv(
    f"{FOLDER_SUMBER}/PDRB_ADHK_Rill.csv", sep=";", decimal=",", encoding="utf-8-sig"
)
pdrb.columns = ["PROVINSI", "year", "pdrb_adhk_rp"]
pdrb["PROVINSI"] = pdrb["PROVINSI"].apply(harmonisasi_provinsi)

deforestasi = pd.read_csv(f"{FOLDER_OUTPUT}/panel_deforestasi_nilai_rupiah_2010_2025.csv")
deforestasi["PROVINSI"] = deforestasi["provinsi"].apply(harmonisasi_provinsi)

pm25 = pd.read_csv(f"{FOLDER_OUTPUT}/panel_nilai_pm25_lengkap_2010_2025.csv")

# --- Verifikasi harmonisasi PDRB & deforestasi ---
for nama, df in [("PDRB", pdrb), ("Deforestasi", deforestasi)]:
    n_tidak_dikenali = df["PROVINSI"].isnull().sum()
    if n_tidak_dikenali > 0:
        print(f"PERINGATAN -- {nama}: {n_tidak_dikenali} baris gagal dipetakan.")
    print(f"[{nama}] {df['PROVINSI'].nunique()} provinsi, "
          f"{df['year'].min()}-{df['year'].max()}, {len(df)} baris")

[PDRB] 34 provinsi, 2010-2025, 544 baris
[Deforestasi] 34 provinsi, 2010-2025, 544 baris


In [ ]:
# =====================================================================
# 3. GABUNGKAN
# =====================================================================
master = pdrb.merge(
    deforestasi[["PROVINSI", "year", "co2e_deforestasi_Mg",
                 "nilai_deforestasi_rp_3pct", "nilai_deforestasi_rp_5pct",
                 "nilai_deforestasi_rp_2_5pct"]],
    on=["PROVINSI", "year"], how="left",
)
master = master.merge(
    pm25[["PROVINSI", "year", "daly_jumlah", "kematian_jumlah",
          "nilai_pm25_rp_daly_1x", "nilai_pm25_rp_daly_3x", "nilai_pm25_rp_vsl",
          "sumber_gbd"]],
    on=["PROVINSI", "year"], how="left",
)

print(f"\nJumlah baris setelah gabung: {len(master)} (harapan 34x16=544)")

# --- Cek baris yang gagal gabung (nilai kosong akibat mismatch nama/tahun) ---
kosong = master[master[["pdrb_adhk_rp", "nilai_deforestasi_rp_3pct", "nilai_pm25_rp_daly_1x"]].isnull().any(axis=1)]
if len(kosong) > 0:
    print(f"\nPERINGATAN -- {len(kosong)} baris punya nilai kosong setelah digabung:")
    print(kosong[["PROVINSI", "year"]].to_string(index=False))
else:
    print("Semua baris lengkap, tidak ada yang gagal digabung.")


Jumlah baris setelah gabung: 544 (harapan 34x16=544)

PERINGATAN -- 4 baris punya nilai kosong setelah digabung:
        PROVINSI  year
KALIMANTAN UTARA  2010
KALIMANTAN UTARA  2011
KALIMANTAN TIMUR  2012
KALIMANTAN UTARA  2012


In [ ]:
# =====================================================================
# 4. HITUNG PDRB HIJAU (tiga skenario)
# =====================================================================
master["pdrb_hijau_utama"] = (
    master["pdrb_adhk_rp"]
    - master["nilai_deforestasi_rp_3pct"]
    - master["nilai_pm25_rp_vsl"]
)
master["pdrb_hijau_sensitivitas_daly_1x"] = (
    master["pdrb_adhk_rp"]
    - master["nilai_deforestasi_rp_3pct"]
    - master["nilai_pm25_rp_daly_1x"]
)
master["pdrb_hijau_sensitivitas_daly_3x"] = (
    master["pdrb_adhk_rp"]
    - master["nilai_deforestasi_rp_2_5pct"]
    - master["nilai_pm25_rp_daly_3x"]
)

# Persentase penyusutan PDRB akibat degradasi lingkungan (skenario utama)
master["persen_penyusutan"] = (
    (master["pdrb_adhk_rp"] - master["pdrb_hijau_utama"]) / master["pdrb_adhk_rp"] * 100
)

In [ ]:
# =====================================================================
# 5. VERIFIKASI KEWAJARAN
# =====================================================================
print(f"\n{'='*70}\nVERIFIKASI KEWAJARAN\n{'='*70}")

# PDRB Hijau tidak boleh negatif -- kalau ada, itu tanda ada yang salah skala
negatif = master[master["pdrb_hijau_utama"] < 0]
if len(negatif) > 0:
    print(f"PERINGATAN -- {len(negatif)} baris PDRB Hijau NEGATIF (skenario utama):")
    print(negatif[["PROVINSI", "year", "pdrb_adhk_rp", "pdrb_hijau_utama"]].to_string(index=False))
else:
    print("Tidak ada PDRB Hijau negatif pada skenario utama -- baik.")

print(f"\nRata-rata persentase penyusutan PDRB (skenario utama): "
      f"{master['persen_penyusutan'].mean():.2f}%")
print(f"Rentang persentase penyusutan: "
      f"{master['persen_penyusutan'].min():.2f}% s.d. {master['persen_penyusutan'].max():.2f}%")

print("\nContoh hasil (Aceh, semua tahun):")
kolom_tampil = ["year", "pdrb_adhk_rp", "nilai_deforestasi_rp_3pct",
                 "nilai_pm25_rp_vsl", "pdrb_hijau_utama", "persen_penyusutan"]
print(master[master["PROVINSI"] == "ACEH"][kolom_tampil].to_string(
    index=False, float_format=lambda x: f"{x:,.0f}"
))

print("\n5 provinsi dengan persentase penyusutan TERBESAR (rata-rata 2010-2025):")
print(master.groupby("PROVINSI")["persen_penyusutan"].mean().sort_values(ascending=False).head())

print("\n5 provinsi dengan persentase penyusutan TERKECIL (rata-rata 2010-2025):")
print(master.groupby("PROVINSI")["persen_penyusutan"].mean().sort_values().head())


VERIFIKASI KEWAJARAN
Tidak ada PDRB Hijau negatif pada skenario utama -- baik.

Rata-rata persentase penyusutan PDRB (skenario utama): 11.55%
Rentang persentase penyusutan: 2.18% s.d. 74.90%

Contoh hasil (Aceh, semua tahun):
 year        pdrb_adhk_rp  nilai_deforestasi_rp_3pct  nilai_pm25_rp_vsl    pdrb_hijau_utama  persen_penyusutan
 2010 101,545,000,000,000          7,263,864,501,541  5,170,087,195,507  89,111,048,302,953                 12
 2011 104,874,000,000,000          9,160,178,905,038  5,163,823,657,935  90,549,997,437,027                 14
 2012 108,915,000,000,000          9,618,643,364,218  5,352,197,693,977  93,944,158,941,805                 14
 2013 111,756,000,000,000          7,384,273,179,691  5,388,422,809,713  98,983,304,010,596                 11
 2014 113,490,000,000,000         11,348,200,273,038  5,348,812,780,290  96,792,986,946,672                 15
 2015 112,666,000,000,000          6,336,152,067,188  5,230,089,160,630 101,099,758,772,183                

In [ ]:
# =====================================================================
# 6. SIMPAN
# =====================================================================
master = master.sort_values(["PROVINSI", "year"]).reset_index(drop=True)
output_path = f"{FOLDER_OUTPUT}/master_panel_pdrb_hijau_2010_2025.csv"
master.to_csv(output_path, index=False)
print(f"\nDisimpan ke: {output_path}")

print(
    "\nCATATAN METODOLOGI:\n"
    "- pdrb_hijau_utama              = PDRB ADHK - deforestasi (SCC 3%) - PM2.5 (VSL, World Bank 2016)\n"
    "  METODE UTAMA, dipakai sbg variabel\n"
    "  dependen dalam estimasi GMM. Diganti dari basis DALY karena VSL lebih\n"
    "  sesuai perkembangan riset terkini (WHO tidak lagi merekomendasikan\n"
    "  konvensi DALYxGDP sejak Bertram dkk. 2016) dan sudah diverifikasi\n"
    "  persis terhadap sumber primer.\n"
    "- pdrb_hijau_sensitivitas_daly_1x = deforestasi (SCC 3%) + PM2.5 (DALY\n"
    "  1x) -- SENSITIVITAS (sebelumnya metode utama, kini pembanding).\n"
    "- pdrb_hijau_sensitivitas_daly_3x = deforestasi (SCC 2,5%) + PM2.5\n"
    "  (DALY 3x) -- sensitivitas skenario kerugian lebih tinggi.\n"
    "- 4 baris (Kalimantan Utara 2010-2012, Kalimantan Timur 2012) bernilai\n"
    "  NaN pada semua skenario akibat gap struktural pemekaran provinsi --\n"
    "  dikecualikan otomatis dari GMM sbg bagian dari unbalanced panel,\n"
    "  konsisten dgn robustness with/without Kalimantan Utara yang sudah\n"
    "  diterapkan pada model."
)


Disimpan ke: /content/drive/MyDrive/DatabaseSkripsi/Outputs/master_panel_pdrb_hijau_2010_2025.csv

CATATAN METODOLOGI:
- pdrb_hijau_utama              = PDRB ADHK - deforestasi (SCC 3%) -
  PM2.5 (VSL, World Bank 2016) -- METODE UTAMA, dipakai sbg variabel
  dependen dalam estimasi GMM. Diganti dari basis DALY karena VSL lebih
  sesuai perkembangan riset terkini (WHO tidak lagi merekomendasikan
  konvensi DALYxGDP sejak Bertram dkk. 2016) dan sudah diverifikasi
  persis terhadap sumber primer.
- pdrb_hijau_sensitivitas_daly_1x = deforestasi (SCC 3%) + PM2.5 (DALY
  1x) -- SENSITIVITAS (sebelumnya metode utama, kini pembanding).
- pdrb_hijau_sensitivitas_daly_3x = deforestasi (SCC 2,5%) + PM2.5
  (DALY 3x) -- sensitivitas skenario kerugian lebih tinggi.
- 4 baris (Kalimantan Utara 2010-2012, Kalimantan Timur 2012) bernilai
  NaN pada semua skenario akibat gap struktural pemekaran provinsi --
  dikecualikan otomatis dari GMM sbg bagian dari unbalanced panel,
  konsisten dgn robustness w